# Action Recognition Inference

This notebook provides inference functionality for action recognition models.
It can load trained models (classifier or contrastive) and predict walking/non-walking actions from JSON keypoint files.

## 1. Import Libraries

In [ ]:
import torch
import json
import numpy as np
import os
from train_action_classifier import ActionClassifier
from train_action_contrastive import ContrastiveActionModel

## 2. Define ActionPredictor Class

In [ ]:
class ActionPredictor:
    """Predict action from keypoints sequence"""
    
    def __init__(self, model_path, model_type='classifier', device='cuda'):
        """
        Args:
            model_path: Path to saved model (.pth file)
            model_type: 'classifier' or 'contrastive'
            device: 'cuda' or 'cpu'
        """
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.model_type = model_type
        
        # Load model architecture (assume 17 keypoints * 3 = 51 features per frame)
        input_size = 51
        
        if model_type == 'classifier':
            self.model = ActionClassifier(
                input_size=input_size,
                hidden_size=128,
                num_layers=2,
                num_classes=2
            )
        else:  # contrastive
            self.model = ContrastiveActionModel(
                input_size=input_size,
                hidden_size=128,
                num_layers=2,
                num_classes=2
            )
        
        # Load weights
        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()
        
        print(f"Loaded {model_type} model from {model_path}")
    
    def preprocess_json(self, json_path, sequence_length=30):
        """Load and preprocess JSON file"""
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        frames = data.get('frames', [])
        
        if len(frames) < sequence_length:
            # Pad if too short
            while len(frames) < sequence_length:
                frames.append(frames[-1] if frames else [])
        
        # Take first sequence_length frames
        frames = frames[:sequence_length]
        
        # Extract keypoints
        keypoints_seq = []
        for frame in frames:
            if isinstance(frame, list) and len(frame) > 0:
                kp = np.array(frame).flatten()
                keypoints_seq.append(kp)
            else:
                # If frame is invalid, use zeros
                kp = np.zeros(51)
                keypoints_seq.append(kp)
        
        keypoints_array = np.array(keypoints_seq, dtype=np.float32)
        
        # Normalize
        keypoints_array[:, 0::3] /= 224.0  # x
        keypoints_array[:, 1::3] /= 224.0  # y
        
        return torch.FloatTensor(keypoints_array).unsqueeze(0)  # Add batch dimension
    
    def predict(self, json_path, return_confidence=False):
        """
        Predict action from JSON file
        
        Returns:
            prediction: 0 (non-walking) or 1 (walking)
            confidence: probability of prediction (if return_confidence=True)
        """
        sequence = self.preprocess_json(json_path).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(sequence)
            probabilities = torch.softmax(outputs, dim=1)
            confidence, predicted = torch.max(probabilities, 1)
        
        prediction = predicted.item()
        conf = confidence.item()
        
        if return_confidence:
            return prediction, conf
        return prediction
    
    def predict_batch(self, json_paths):
        """Predict multiple files at once"""
        results = []
        for path in json_paths:
            try:
                pred, conf = self.predict(path, return_confidence=True)
                action = 'Walking' if pred == 1 else 'Non-Walking'
                results.append({
                    'file': os.path.basename(path),
                    'prediction': action,
                    'confidence': f"{conf:.2%}"
                })
            except Exception as e:
                results.append({
                    'file': os.path.basename(path),
                    'prediction': 'Error',
                    'confidence': str(e)
                })
        return results

## 3. Example Usage - Single File Prediction

In [ ]:
# Example: Load classifier model and predict single file
model_path = 'action_classifier_best.pth'
json_file = 'dataset/walking/video1.json'

# Create predictor
predictor = ActionPredictor(model_path, model_type='classifier')

# Make prediction
pred, conf = predictor.predict(json_file, return_confidence=True)
action = 'Walking' if pred == 1 else 'Non-Walking'

print(f"Prediction: {action}")
print(f"Confidence: {conf:.2%}")

## 4. Example Usage - Batch Prediction

In [ ]:
# Example: Process folder of JSON files
folder_path = 'dataset/walking/'

# Get all JSON files in folder
json_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) 
             if f.endswith('.json')]

print(f"Processing {len(json_files)} files...\n")

# Batch prediction
results = predictor.predict_batch(json_files)

# Display results
print(f"{'File':<40} {'Prediction':<15} {'Confidence'}")
print("="*70)
for r in results:
    print(f"{r['file']:<40} {r['prediction']:<15} {r['confidence']}")